In [1]:
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '_detected_manual' # '', _detected', '_detected_manual'

In [2]:
import torch

# Load the saved embeddings
embeddings_path = f'saved_models/{megadescriptor_version}/embeddings/emb{detection}.pt'
labels_path = f'saved_models/{megadescriptor_version}/labels/labels{detection}.pt'
label_encoder = f'saved_models/{megadescriptor_version}/label_encoders/label_encoder{detection}.pkl'

embeddings = torch.load(embeddings_path)

print(embeddings.shape)  # torch.Size([260, 768])


torch.Size([315, 768])


In [3]:
import torch, joblib
label_ids = torch.load(labels_path, weights_only=False)
encoder = joblib.load(label_encoder)

# Convert names back later:
names = encoder.inverse_transform(label_ids)


In [4]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Example setup
X = embeddings.float()
y = torch.from_numpy(label_ids).long()   # shape (260,)

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [5]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [6]:
from proportional_split_xy import proportional_split_xy

# Triplet Loss


In [7]:
margin = 0.85

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


### Hard split

In [9]:
import torch
import numpy as np

def generate_hard_triplets(embeddings, labels, num_triplets_per_anchor=20, margin=1.0, device=None):
    """
    Hard triplet mining: for each anchor, pick hardest positive and hardest negative.

    Args:
        embeddings (torch.Tensor): shape (N, D)
        labels (torch.Tensor | np.ndarray): shape (N,)
        num_triplets_per_anchor (int): how many copies to make per anchor
        margin (float): triplet loss margin
        device (torch.device): e.g., torch.device('cuda') or ('cpu')
    """
    if device is None:
        device = torch.device("cpu")

    # ensure tensor
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    embeddings = embeddings.to(device)
    labels = labels.to(device)
    triplets = []

    # normalize embeddings for cosine-style distance
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    # pairwise Euclidean distances
    with torch.no_grad():
        dists = torch.cdist(embeddings, embeddings, p=2)

    labels_np = labels.cpu().numpy()
    n = len(labels_np)

    for anchor_idx in range(n):
        anchor_label = labels_np[anchor_idx]

        pos_indices = np.where(labels_np == anchor_label)[0]
        neg_indices = np.where(labels_np != anchor_label)[0]

        if len(pos_indices) < 2:
            continue  # skip if only one image of this class

        # hardest positive = most distant same-class example
        pos_dists = dists[anchor_idx, pos_indices]
        hardest_pos_idx = pos_indices[torch.argmax(pos_dists)].item()

        # hardest negative = closest different-class example
        neg_dists = dists[anchor_idx, neg_indices]
        hardest_neg_idx = neg_indices[torch.argmin(neg_dists)].item()

        # replicate triplet if needed
        for _ in range(num_triplets_per_anchor):
            triplets.append((anchor_idx, hardest_pos_idx, hardest_neg_idx))

    print(f"✅ Generated {len(triplets)} hard triplets on device {device}.")
    return triplets


### Semi hard split

In [10]:
def generate_semi_hard_triplets(embeddings, labels, num_triplets_per_anchor=20, margin=1.0, device=None):
    """
    Semi-hard triplet mining:
    For each anchor, choose positive that is closer than negative but still violates the margin slightly:
        d(a, p) < d(a, n) < d(a, p) + margin

    embeddings: torch.Tensor (N, D)
    labels: torch.Tensor (N,)
    """
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)
    if device is not None:
        embeddings = embeddings.to(device)
        labels = labels.to(device)

    triplets = []
    num_samples = len(labels)

    # Precompute pairwise distances
    dists = torch.cdist(embeddings, embeddings, p=2).detach().cpu()

    labels_np = labels.cpu().numpy()

    for anchor_idx in range(num_samples):
        anchor_label = labels_np[anchor_idx]

        pos_indices = np.where(labels_np == anchor_label)[0]
        neg_indices = np.where(labels_np != anchor_label)[0]
        if len(pos_indices) < 2 or len(neg_indices) == 0:
            continue

        # Hardest positive (furthest same-class)
        d_ap_all = dists[anchor_idx, pos_indices]
        d_ap_all = d_ap_all[d_ap_all > 0]  # exclude self
        if len(d_ap_all) == 0:
            continue

        for _ in range(num_triplets_per_anchor):
            pos_idx = np.random.choice(pos_indices)
            d_ap = dists[anchor_idx, pos_idx]

            # Find semi-hard negatives
            d_an_candidates = dists[anchor_idx, neg_indices]
            mask = (d_an_candidates > d_ap) & (d_an_candidates < d_ap + margin)
            valid_negatives = neg_indices[mask.numpy()]

            if len(valid_negatives) == 0:
                # fallback to hardest negative
                neg_idx = neg_indices[d_an_candidates.argmin().item()]
            else:
                neg_idx = np.random.choice(valid_negatives)

            triplets.append((anchor_idx, pos_idx, neg_idx))

    print(f"Generated {len(triplets)} semi-hard triplets.")
    return triplets


### With indices


In [11]:
def generate_semi_hard_triplets(
    embeddings,
    labels,
    indices=None,                 # NEW
    num_triplets_per_anchor=20,
    margin=1.0,
    device=None
):
    """
    Semi-hard triplet mining on a subset of indices.

    embeddings: (N, D)
    labels: (N,)
    indices: list/array of sample indices to use as anchors
    """

    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    if device is not None:
        embeddings = embeddings.to(device)
        labels = labels.to(device)

    if indices is None:
        indices = np.arange(len(labels))
    else:
        indices = np.array(indices)

    triplets = []

    # Compute global pairwise distances once
    dists = torch.cdist(embeddings, embeddings, p=2).detach().cpu()
    labels_np = labels.cpu().numpy()

    for anchor_idx in indices:
        anchor_label = labels_np[anchor_idx]

        pos_indices = np.where(labels_np == anchor_label)[0]
        neg_indices = np.where(labels_np != anchor_label)[0]

        # remove self from positives
        pos_indices = pos_indices[pos_indices != anchor_idx]

        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue

        for _ in range(num_triplets_per_anchor):

            pos_idx = np.random.choice(pos_indices)
            d_ap = dists[anchor_idx, pos_idx]

            d_an_candidates = dists[anchor_idx, neg_indices]
            mask = (d_an_candidates > d_ap) & (d_an_candidates < d_ap + margin)
            valid_negatives = neg_indices[mask.numpy()]

            if len(valid_negatives) == 0:
                # fallback: hardest negative
                neg_idx = neg_indices[d_an_candidates.argmin().item()]
            else:
                neg_idx = np.random.choice(valid_negatives)

            triplets.append((anchor_idx, pos_idx, neg_idx))

    print(f"Generated {len(triplets)} semi-hard triplets.")
    return triplets


### Random split

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict

# ------------------------------
# Model: scales each embedding dimension independently
# ------------------------------
class ScaledEmbeddingModel(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(emb_dim))
    def forward(self, x):
        return x * self.scale


# ------------------------------
# Compute accuracy: fraction of triplets where anchor-positive < anchor-negative
# ------------------------------
def compute_triplet_accuracy(model, embeddings, triplets):
    correct = 0
    with torch.no_grad():
        for (a, p, n) in triplets:
            anchor = model(embeddings[a].unsqueeze(0))
            positive = model(embeddings[p].unsqueeze(0))
            negative = model(embeddings[n].unsqueeze(0))
            dist_pos = torch.norm(anchor - positive, p=2)
            dist_neg = torch.norm(anchor - negative, p=2)
            if dist_pos < dist_neg:
                correct += 1
    return correct / len(triplets)


# ------------------------------
# Stratified random split of triplets
# - ensures disjoint train/test triplets
# - ensures each class (anchor label) appears in both sets
# ------------------------------
import numpy as np
import torch
import random
from collections import defaultdict


### Stratified index split

In [13]:
def stratified_index_split(labels, test_ratio=0.2, seed=42):
    """
    Stratified split of sample indices.

    Ensures:
    - Disjoint train/val indices
    - Every class appears in both sets (if class size > 1)

    labels: array-like of shape (N,)
    """

    random.seed(seed)
    np.random.seed(seed)

    labels = np.array(labels)
    unique_classes = np.unique(labels)

    train_indices = []
    val_indices = []

    for cls in unique_classes:
        cls_indices = np.where(labels == cls)[0]
        np.random.shuffle(cls_indices)

        n_total = len(cls_indices)
        n_val = max(1, int(n_total * test_ratio))

        if n_total == 1:
            # cannot split
            train_indices.extend(cls_indices)
        else:
            val_indices.extend(cls_indices[:n_val])
            train_indices.extend(cls_indices[n_val:])

    train_indices = np.array(train_indices)
    val_indices = np.array(val_indices)

    print("========== STRATIFIED INDEX SPLIT ==========")
    print("Train size:", len(train_indices))
    print("Val size:", len(val_indices))
    print("Overlap:", len(set(train_indices) & set(val_indices)))

    return train_indices, val_indices


In [14]:
# ------------------------------
# Training loop
# ------------------------------
def train_triplet_loss(embeddings, train_triplets, val_triplets,
                       emb_dim=128, lr=1e-3, margin=1.0, epochs=50):
    model = ScaledEmbeddingModel(emb_dim)
    criterion = nn.TripletMarginLoss(margin=margin)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        if epoch != 0:
            model.train()
        total_loss = 0.0

        for (a, p, n) in train_triplets:
            anchor = model(embeddings[a].unsqueeze(0))
            positive = model(embeddings[p].unsqueeze(0))
            negative = model(embeddings[n].unsqueeze(0))

            loss = criterion(anchor, positive, negative)
            if epoch != 0:                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()

        model.eval()
        train_acc = compute_triplet_accuracy(model, embeddings, train_triplets)
        val_acc = compute_triplet_accuracy(model, embeddings, val_triplets)

        print(f"Epoch {epoch:02d} | loss: {total_loss/len(train_triplets):.6f} | "
              f"train_acc: {train_acc:.3f} | val_acc: {val_acc:.3f}")

    print("\n✅ Final Validation Accuracy:", 
          round(compute_triplet_accuracy(model, embeddings, val_triplets), 4))
    return model

In [15]:
def generate_triplets(indices, labels, num_triplets_per_anchor=20):
    triplets = []
    # Handle both torch.Tensor and np.ndarray
    labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.array(labels)

    for anchor_idx in indices:
        anchor_label = labels_np[anchor_idx]

        # Positive samples (same class, excluding anchor)
        pos_indices = [i for i in indices if labels_np[i] == anchor_label and i != anchor_idx]
        # Negative samples (different class)
        neg_indices = [i for i in indices if labels_np[i] != anchor_label]

        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue

        for _ in range(num_triplets_per_anchor):
            p = random.choice(pos_indices)
            n = random.choice(neg_indices)
            triplets.append((anchor_idx, p, n))

    return triplets

### Main

In [20]:
if __name__ == "__main__":
    import random
    import numpy as np

    torch.manual_seed(0)
    random.seed(0)

    # --- Load your embeddings and labels ---
    
    labels = torch.load(labels_path, weights_only=False)               # shape (N,)
    N, D = embeddings.shape

    # --- Split into train/val indices ---
    indices = np.arange(N)
    np.random.shuffle(indices)
    split = int(0.8 * N)
    train_idx = indices[:split]
    val_idx = indices[split:]




    # --- Generate triplets for train and val ---
    # train_triplets = generate_triplets(train_idx, labels, num_triplets_per_anchor=50)
    # val_triplets = generate_triplets(val_idx, labels, num_triplets_per_anchor=50)

    # all_triplets = generate_semi_hard_triplets(embeddings, labels, num_triplets_per_anchor=20)
    train_indices, val_indices = stratified_index_split(labels, test_ratio=0.2)

    train_triplets = generate_semi_hard_triplets(embeddings, labels, indices=train_indices)
    val_triplets = generate_semi_hard_triplets(embeddings, labels, indices=val_indices)


    # train_triplets = generate_hard_triplets(embeddings[train_idx], labels[train_idx], num_triplets_per_anchor=150, margin=1.1, device=device)
    # val_triplets   = generate_hard_triplets(embeddings[val_idx], labels[val_idx], num_triplets_per_anchor=50, margin=1.0, device=device)

    # train_triplets = generate_semi_hard_triplets(embeddings[train_idx], labels[train_idx], num_triplets_per_anchor=20, margin=1.1, device=device)
    # val_triplets   = generate_semi_hard_triplets(embeddings[val_idx], labels[val_idx], num_triplets_per_anchor=20, margin=1.0, device=device)

    print(f"Generated {len(train_triplets)} training triplets and {len(val_triplets)} validation triplets.")


========== STRATIFIED INDEX SPLIT ==========
Train size: 258
Val size: 57
Overlap: 0
Generated 5160 semi-hard triplets.
Generated 1140 semi-hard triplets.
Generated 5160 training triplets and 1140 validation triplets.


In [24]:
verify_triplet_split(train_triplets, val_triplets)


========== VERIFYING TRIPLET SPLIT ==========
Train unique indices: 315
Val unique indices:   315
Overlap indices:      315
❌ WARNING: Overlapping indices detected!
Example overlapping indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Overlapping triplets: 0
✅ No exact triplet overlap



In [26]:

# --- Train model ---
model = train_triplet_loss(
    embeddings, train_triplets, val_triplets,
    emb_dim=D, lr=1e-3, epochs=10
)

# --- Evaluate final accuracy ---
final_val_acc = compute_triplet_accuracy(model, embeddings, val_triplets)
print(f"\n✅ Final Validation Accuracy: {final_val_acc:.3f}")

# --- Optionally save calibrated embeddings ---
calibrated_embeddings = model(embeddings)
torch.save(calibrated_embeddings, "emb_calibrated.pt")


Epoch 00 | loss: 0.736575 | train_acc: 0.987 | val_acc: 0.987
Epoch 01 | loss: 0.671796 | train_acc: 0.922 | val_acc: 0.909
Epoch 02 | loss: 0.610298 | train_acc: 0.919 | val_acc: 0.893
Epoch 03 | loss: 0.588387 | train_acc: 0.919 | val_acc: 0.882


KeyboardInterrupt: 

In [22]:
def verify_triplet_split(train_triplets, val_triplets):
    print("\n========== VERIFYING TRIPLET SPLIT ==========")

    # ---- 1. Index overlap ----
    train_indices = set(i for t in train_triplets for i in t)
    val_indices   = set(i for t in val_triplets for i in t)

    overlap_indices = train_indices & val_indices

    print(f"Train unique indices: {len(train_indices)}")
    print(f"Val unique indices:   {len(val_indices)}")
    print(f"Overlap indices:      {len(overlap_indices)}")

    if overlap_indices:
        print("❌ WARNING: Overlapping indices detected!")
        print("Example overlapping indices:", list(overlap_indices)[:10])
    else:
        print("✅ No index overlap")

    # ---- 2. Triplet overlap ----
    train_set = set(train_triplets)
    val_set   = set(val_triplets)

    overlapping_triplets = train_set & val_set

    print(f"\nOverlapping triplets: {len(overlapping_triplets)}")

    if overlapping_triplets:
        print("❌ Exact duplicate triplets found!")
        print("Example overlapping triplets:", list(overlapping_triplets)[:5])
    else:
        print("✅ No exact triplet overlap")

    print("============================================\n")


In [25]:
train_triplets

[(4, 1, 228),
 (4, 2, 201),
 (4, 0, 249),
 (4, 0, 36),
 (4, 0, 76),
 (4, 0, 132),
 (4, 3, 55),
 (4, 2, 169),
 (4, 0, 219),
 (4, 3, 64),
 (4, 2, 169),
 (4, 1, 232),
 (4, 1, 137),
 (4, 1, 62),
 (4, 1, 130),
 (4, 2, 169),
 (4, 2, 124),
 (4, 0, 190),
 (4, 0, 177),
 (4, 2, 28),
 (2, 0, 213),
 (2, 3, 71),
 (2, 3, 207),
 (2, 4, 23),
 (2, 4, 254),
 (2, 1, 197),
 (2, 0, 67),
 (2, 3, 138),
 (2, 0, 219),
 (2, 1, 40),
 (2, 4, 23),
 (2, 4, 280),
 (2, 0, 314),
 (2, 4, 161),
 (2, 0, 8),
 (2, 4, 41),
 (2, 0, 135),
 (2, 3, 284),
 (2, 1, 40),
 (2, 4, 277),
 (0, 4, 151),
 (0, 4, 224),
 (0, 3, 11),
 (0, 3, 111),
 (0, 2, 224),
 (0, 1, 125),
 (0, 1, 133),
 (0, 3, 305),
 (0, 4, 240),
 (0, 1, 171),
 (0, 3, 14),
 (0, 2, 218),
 (0, 4, 268),
 (0, 3, 19),
 (0, 1, 23),
 (0, 4, 164),
 (0, 3, 173),
 (0, 4, 151),
 (0, 2, 207),
 (0, 3, 305),
 (3, 2, 266),
 (3, 1, 314),
 (3, 0, 101),
 (3, 1, 288),
 (3, 1, 189),
 (3, 4, 234),
 (3, 1, 168),
 (3, 1, 87),
 (3, 4, 174),
 (3, 0, 132),
 (3, 0, 232),
 (3, 4, 246),
 (3, 0, 111)

In [27]:
val_triplets

[(1, 2, 99),
 (1, 0, 117),
 (1, 3, 225),
 (1, 4, 287),
 (1, 2, 201),
 (1, 4, 280),
 (1, 3, 225),
 (1, 3, 128),
 (1, 2, 99),
 (1, 3, 143),
 (1, 3, 236),
 (1, 4, 113),
 (1, 2, 201),
 (1, 2, 201),
 (1, 0, 33),
 (1, 4, 40),
 (1, 0, 227),
 (1, 0, 64),
 (1, 3, 256),
 (1, 3, 187),
 (22, 6, 273),
 (22, 24, 223),
 (22, 29, 137),
 (22, 27, 298),
 (22, 14, 74),
 (22, 32, 281),
 (22, 13, 72),
 (22, 33, 81),
 (22, 21, 177),
 (22, 14, 187),
 (22, 34, 63),
 (22, 10, 112),
 (22, 16, 58),
 (22, 18, 105),
 (22, 14, 130),
 (22, 25, 145),
 (22, 36, 92),
 (22, 14, 75),
 (22, 33, 176),
 (22, 7, 260),
 (18, 20, 127),
 (18, 16, 206),
 (18, 8, 236),
 (18, 33, 241),
 (18, 27, 141),
 (18, 31, 43),
 (18, 21, 124),
 (18, 23, 73),
 (18, 31, 79),
 (18, 10, 263),
 (18, 28, 212),
 (18, 13, 109),
 (18, 17, 138),
 (18, 7, 48),
 (18, 38, 289),
 (18, 36, 312),
 (18, 39, 4),
 (18, 20, 132),
 (18, 37, 224),
 (18, 26, 43),
 (9, 37, 2),
 (9, 33, 75),
 (9, 20, 111),
 (9, 20, 225),
 (9, 14, 108),
 (9, 25, 62),
 (9, 31, 244),
 (